# 🚀 Модуль 4. Урок 1. Грязные данные: пропуски (MCAR/MAR/MNAR) и приведение типов

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_04_data_cleaning/lesson_01_dirty_data_types_missing_values.ipynb)

## 📌 Бизнес-контекст и методология
В реальном бизнесе 80% времени уходит на очистку данных. Пропуски делятся на:
* **MCAR** (Missing Completely at Random) — случайный сбой датчика
* **MAR** (Missing at Random) — пропуск зависит от других наблюдаемых полей
* **MNAR** (Missing Not at Random) — пропуск зависит от самого значения (клиенты с высоким доходом скрывают зарплату)

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd
import numpy as np

df_saas = pd.read_csv(saas_csv)

print("Аудит пропущенных значений в saas_users.csv:")
missing_report = pd.DataFrame({
    'missing_count': df_saas.isna().sum(),
    'missing_pct': (df_saas.isna().mean() * 100).round(2)
})
print(missing_report[missing_report['missing_count'] > 0])


Аудит пропущенных значений в saas_users.csv:
            missing_count  missing_pct
churn_date           1664         83.2


### 🎯 Практическое задание 1: Корректная обработка пропусков в churn_date
Колонка `churn_date` пуста для активных клиентов (`is_churned = 0`).
Заполните `churn_date` для активных клиентов значением 'Still_Active' и убедитесь, что пропусков не осталось.

In [3]:
### ВАШ КОД ЗДЕСЬ
df_clean = df_saas.copy()
df_clean['churn_date_filled'] = df_clean['churn_date'].fillna('Still_Active')

assert df_clean['churn_date_filled'].isna().sum() == 0, "Пропусков не должно остаться"
print("✅ Тест успешно пройден: пропуски обработаны осмысленно!")


✅ Тест успешно пройден: пропуски обработаны осмысленно!


### 💡 Ловушка среднего при заполнении пропусков
Заполнение пропусков средним (`df.fillna(df.mean())`) искусственно занижает дисперсию и искажает доверительные интервалы.